# Daltonize: Color Blindness Simulation and Correction

This notebook demonstrates the Daltonize algorithm for simulating color blindness and correcting images for color-blind viewers.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

# Import the DaltonizeLayer (assuming it's accessible from daltonize_algorithm.ipynb)
# For demonstration, we'll define it inline or import from the module
import sys
sys.path.insert(0, '/content')

# Since we're in a notebook environment, we'll use the following workaround:
# Copy the necessary code or import from the algorithm notebook

# For now, let's define the matrices and layer here for demonstration
# In production, these would be imported from daltonize_algorithm.ipynb

print("TensorFlow version:", tf.__version__)
print("Imports successful!")

## Load and Prepare Image

Load a sample image and convert it to a TensorFlow tensor with values normalized to [0, 1].

In [ ]:
# Load a sample image using scikit-image
# Using the 'coffee' image as an example
image = data.coffee()

# Normalize to [0, 1]
image_normalized = image.astype(np.float32) / 255.0

# Add batch dimension: (1, Height, Width, 3)
image_batch = tf.expand_dims(image_normalized, 0)

print(f"Image shape: {image_batch.shape}")
print(f"Image dtype: {image_batch.dtype}")
print(f"Image value range: [{tf.reduce_min(image_batch):.3f}, {tf.reduce_max(image_batch):.3f}]")

## Define Daltonize Matrices and Layer

Since we're in a separate notebook, we'll define the transformation matrices and the DaltonizeLayer class here.

In [ ]:
# Define transformation matrices
RGB_TO_LMS = tf.constant([
    [0.3, 0.622, 0.078],
    [0.23, 0.692, 0.078],
    [0.25, 0.125, 0.625]
], dtype=tf.float32)

LMS_TO_RGB = tf.constant([
    [11.031, -9.38, -0.651],
    [-3.254, 2.414, -0.16],
    [-3.66, 3.283, 0.377]
], dtype=tf.float32)

# Deuteranopia matrices
DEUT_SIM_MATRIX = tf.constant([
    [1.0, 0.0, 0.0],
    [0.494207, 0.0, 1.24827],
    [0.0, 0.0, 1.0]
], dtype=tf.float32)

DEUT_CORRECTION_MATRIX = tf.constant([
    [1.0, 0.0, 0.0],
    [-0.882516, 0.0, -0.805306],
    [0.0, 0.0, 1.0]
], dtype=tf.float32)

# Protanopia matrices
PROT_SIM_MATRIX = tf.constant([
    [0.0, 2.02344, -2.52581],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
], dtype=tf.float32)

PROT_CORRECTION_MATRIX = tf.constant([
    [0.0, -0.4942, 1.1945],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
], dtype=tf.float32)

# Tritanopia matrices
TRIT_SIM_MATRIX = tf.constant([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [-0.395913, 0.801109, 0.0]
], dtype=tf.float32)

TRIT_CORRECTION_MATRIX = tf.constant([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.494207, -1.000000, 0.0]
], dtype=tf.float32)

print("Transformation matrices defined.")

In [ ]:
class DaltonizeLayer(tf.keras.layers.Layer):
    """
    Custom Keras layer for simulating or correcting color blindness.
    
    Args:
        deficiency_type (str): Type of color blindness ('protanopia', 'deuteranopia', 'tritanopia')
        correction (bool): If True, apply correction; if False, apply simulation (default=False)
    """
    
    def __init__(self, deficiency_type='deuteranopia', correction=False, **kwargs):
        super(DaltonizeLayer, self).__init__(**kwargs)
        self.deficiency_type = deficiency_type.lower()
        self.correction = correction
        
        # Select the appropriate matrices
        if self.deficiency_type == 'protanopia':
            self.sim_matrix = PROT_SIM_MATRIX
            self.corr_matrix = PROT_CORRECTION_MATRIX
        elif self.deficiency_type == 'deuteranopia':
            self.sim_matrix = DEUT_SIM_MATRIX
            self.corr_matrix = DEUT_CORRECTION_MATRIX
        elif self.deficiency_type == 'tritanopia':
            self.sim_matrix = TRIT_SIM_MATRIX
            self.corr_matrix = TRIT_CORRECTION_MATRIX
        else:
            raise ValueError(f"Unknown deficiency type: {deficiency_type}")
        
        # Select the transformation matrix based on mode
        self.transform_matrix = self.corr_matrix if self.correction else self.sim_matrix
    
    def call(self, inputs):
        """
        Apply Daltonize transformation to input images.
        
        Args:
            inputs: Tensor of shape (Batch, Height, Width, 3) with values in [0, 1]
        
        Returns:
            Transformed image tensor of the same shape
        """
        # Store original shape and flatten spatial dimensions
        original_shape = tf.shape(inputs)
        batch_size = original_shape[0]
        height = original_shape[1]
        width = original_shape[2]
        
        # Reshape to (Batch * Height * Width, 3) for matrix multiplication
        flat_images = tf.reshape(inputs, [-1, 3])
        
        # Step 1: Convert sRGB to LMS
        lms = tf.matmul(flat_images, tf.transpose(RGB_TO_LMS))
        
        # Step 2: Apply simulation or correction matrix
        transformed_lms = tf.matmul(lms, tf.transpose(self.transform_matrix))
        
        # Step 3: Convert back from LMS to sRGB
        output = tf.matmul(transformed_lms, tf.transpose(LMS_TO_RGB))
        
        # Reshape back to original image dimensions
        output = tf.reshape(output, original_shape)
        
        # Clamp values to [0, 1] to maintain valid image range
        output = tf.clip_by_value(output, 0.0, 1.0)
        
        return output
    
    def get_config(self):
        """Return config for serialization."""
        config = super().get_config()
        config.update({
            'deficiency_type': self.deficiency_type,
            'correction': self.correction
        })
        return config

print("DaltonizeLayer class defined.")

## Simulate Deuteranopia (Green Blindness)

Process the original image through the Deuteranopia simulation layer.

In [ ]:
# Create Deuteranopia simulation layer
deut_sim_layer = DaltonizeLayer(deficiency_type='deuteranopia', correction=False)

# Apply simulation to the original image
simulated_image = deut_sim_layer(image_batch)

# Convert tensor to numpy for visualization
simulated_image_np = simulated_image[0].numpy()

print(f"Simulated image shape: {simulated_image_np.shape}")
print(f"Simulated image value range: [{simulated_image_np.min():.3f}, {simulated_image_np.max():.3f}]")

## Correct for Deuteranopia (Daltonize)

Apply color correction to make the original image visible to color-blind viewers.

In [ ]:
# Create Deuteranopia correction layer
deut_corr_layer = DaltonizeLayer(deficiency_type='deuteranopia', correction=True)

# Apply correction to the original image
corrected_image = deut_corr_layer(image_batch)

# Convert tensor to numpy for visualization
corrected_image_np = corrected_image[0].numpy()

print(f"Corrected image shape: {corrected_image_np.shape}")
print(f"Corrected image value range: [{corrected_image_np.min():.3f}, {corrected_image_np.max():.3f}]")

## Visualization

Display the original, simulated, and corrected images side-by-side.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original image
original_image_np = image_normalized
axes[0].imshow(original_image_np)
axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
axes[0].axis('off')

# Simulated Deuteranopia
axes[1].imshow(simulated_image_np)
axes[1].set_title('Deuteranopia Simulation\n(What a color-blind person sees)', fontsize=14, fontweight='bold')
axes[1].axis('off')

# Corrected (Daltonized)
axes[2].imshow(corrected_image_np)
axes[2].set_title('Daltonized (Corrected)\n(For color-blind viewers)', fontsize=14, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("Visualization complete!")

## Experiment with Other Deficiency Types

Try simulating and correcting for Protanopia (red blindness) and Tritanopia (blue blindness).

In [ ]:
# Simulate Protanopia (red blindness)
prot_sim_layer = DaltonizeLayer(deficiency_type='protanopia', correction=False)
prot_sim_image = prot_sim_layer(image_batch)[0].numpy()

# Simulate Tritanopia (blue blindness)
trit_sim_layer = DaltonizeLayer(deficiency_type='tritanopia', correction=False)
trit_sim_image = trit_sim_layer(image_batch)[0].numpy()

# Display all three simulations
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(original_image_np)
axes[0].set_title('Original', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(prot_sim_image)
axes[1].set_title('Protanopia\n(Red Blindness)', fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(simulated_image_np)
axes[2].set_title('Deuteranopia\n(Green Blindness)', fontsize=12, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(trit_sim_image)
axes[3].set_title('Tritanopia\n(Blue Blindness)', fontsize=12, fontweight='bold')
axes[3].axis('off')

plt.tight_layout()
plt.show()

print("All color blindness simulations complete!")